Một chiến lược mua: Khi giá dự đoán tăng so với giá hiện tại, Momentum dương (điều này cho thấy xu hướng giá đang tăng), và RSI dưới 70 (điều này cho thấy rằng cổ phiếu không bị mua quá mức).

Một chiến lược bán: Khi giá dự đoán giảm so với giá hiện tại, Momentum âm (điều này cho thấy xu hướng giá đang giảm), và RSI trên 30 (điều này cho thấy cổ phiếu không bị bán quá mức).

# Load du lieu len

In [2]:
import sys
sys.path.append('../Common')

import CommonMT5
import MetaTrader5 as mt5
from datetime import datetime, timedelta

symbol = 'BTCUSDs'
from_date = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua
to_date = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')
timeframe = mt5.TIMEFRAME_M1

data = CommonMT5.CommonMT5.loaddataMT5_FromTo(symbol, from_date, to_date, timeframe)

data

,Datetime,Open,High,Low,Close,Volume
0,2024-11-27 17:00:00,94861.61,94920.55,94834.45,94863.05,162
1,2024-11-27 17:01:00,94835.03,94862.95,94718.56,94718.66,228
2,2024-11-27 17:02:00,94718.67,94786.45,94656.64,94775.05,193
3,2024-11-27 17:03:00,94774.64,94793.95,94697.75,94793.05,207
4,2024-11-27 17:04:00,94793.40,94925.19,94793.15,94874.65,137
...,...,...,...,...,...,...
2866,2024-11-29 16:56:00,97866.45,98013.95,97831.85,97974.05,126
2867,2024-11-29 16:57:00,97964.75,98013.30,97951.65,97972.45,92
2868,2024-11-29 16:58:00,97963.25,98000.85,97915.95,97978.75,91
2869,2024-11-29 16:59:00,97978.85,98112.05,97978.85,98049.55,60


# Tim tham so p, d, q toi uu: Tim 1 lan thoi

In [3]:
# import itertools
# import statsmodels.api as sm
# import pandas as pd

# # Đặt chỉ mục DataFrame với cột 'Datetime' có dữ liệu kiểu datetime
# # data.set_index('Datetime', inplace=True) da dat o tren

# # Xác định khoảng giá trị cho tham số p, d, q
# p = d = q = range(0, 6)
# pdq = list(itertools.product(p, d, q))

# best_aic = float("inf")
# best_pdq = None
# best_model = None

# data['MA'] = data['Close'].rolling(window=5).mean().dropna()  # Ví dụ dùng cửa sổ trượt kích thước 5

# for param in pdq:
#     try:
#         model = sm.tsa.ARIMA(data['MA'], order=param)
#         results = model.fit()
#         if results.aic < best_aic:
#             best_aic = results.aic
#             best_pdq = param
#             best_model = results
#     except:
#         continue

# print(f'Best ARIMA{best_pdq} AIC:{best_aic}')


c:\Users\PC-DELL-CU\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
c:\Users\PC-DELL-CU\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
c:\Users\PC-DELL-CU\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
c:\Users\PC-DELL-CU\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as

# Tinh toan chien luoc

In [3]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.pyplot as plt
import sys
import talib

# Giả sử `data` là DataFrame chứa các cột Datetime, Open, High, Low, Close, Volume
# Đặt chỉ mục DataFrame với cột 'Datetime' có dữ liệu kiểu datetime
data.set_index('Datetime', inplace=True)

# Tính Momentum
data['Momentum'] = data['Close'].diff()

# Tính RSI
data['RSI'] = talib.RSI(data['Close'], timeperiod=14)

# Chọn tham số ARIMA dựa trên AIC, BIC hoặc các tiêu chí khác (cần phải thực hiện trước) => Chay cell o tren
model = ARIMA(data['Close'], order=(5, 2, 4))
model_fit = model.fit()

# Dự đoán giá
data['Predicted_Close'] = model_fit.predict(start=0, end=len(data)-1, typ='levels')

# Xác định tín hiệu mua và bán
data['Buy_Signal'] = ((data['Predicted_Close'] > data['Close']) & (data['Momentum'] > -1) & (data['RSI'] < 80))
data['Sell_Signal'] = ((data['Predicted_Close'] < data['Close']) & (data['Momentum'] < 1201) & (data['RSI'] > 30))

# Trực quan hóa
import plotly.graph_objects as go

# Tạo biểu đồ cơ bản với giá đóng cửa
fig = go.Figure()
fig.add_trace(go.Scatter(x=data.index, y=data['Close'], mode='lines', name='Close'))

# Thêm dữ liệu giá dự đoán vào biểu đồ
fig.add_trace(go.Scatter(x=data.index, y=data['Predicted_Close'], mode='lines', name='Predicted Close', line=dict(dash='dot', color='red')))

# Thêm điểm mua
buy_signals = data[data['Buy_Signal']]
fig.add_trace(go.Scatter(x=buy_signals.index, y=buy_signals['Close'], mode='markers', name='Buy Signal', marker=dict(color='green', size=10, symbol='triangle-up')))

# Thêm điểm bán
sell_signals = data[data['Sell_Signal']]
fig.add_trace(go.Scatter(x=sell_signals.index, y=sell_signals['Close'], mode='markers', name='Sell Signal', marker=dict(color='red', size=10, symbol='triangle-down')))

# Tùy chỉnh layout
fig.update_layout(title='Trading Signals with ARIMA, Momentum, and RSI', xaxis_title='Date', yaxis_title='Price', hovermode='x')

# Hiển thị biểu đồ
fig.show()

C:\Users\dongn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\dongn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\dongn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g

In [4]:
data

,Open,High,Low,Close,Volume,Momentum,RSI,Predicted_Close,Buy_Signal,Sell_Signal
Datetime,,,,,,,,,,
2024-11-27 17:00:00,94861.61,94920.55,94834.45,94863.05,162,NaN,NaN,0.000000,False,False
2024-11-27 17:01:00,94835.03,94862.95,94718.56,94718.66,228,-144.39,NaN,142294.423296,False,False
2024-11-27 17:02:00,94718.67,94786.45,94656.64,94775.05,193,56.39,NaN,94876.781517,False,False
2024-11-27 17:03:00,94774.64,94793.95,94697.75,94793.05,207,18.00,NaN,94875.786525,False,False
2024-11-27 17:04:00,94793.40,94925.19,94793.15,94874.65,137,81.60,NaN,94891.252538,False,False
...,...,...,...,...,...,...,...,...,...,...
2024-11-29 16:56:00,97866.45,98013.95,97831.85,97974.05,126,107.42,73.630018,97870.243113,False,True
2024-11-29 16:57:00,97964.75,98013.30,97951.65,97972.45,92,-1.60,73.475704,97974.048778,False,False
2024-11-29 16:58:00,97963.25,98000.85,97915.95,97978.75,91,6.30,73.709350,97973.903815,False,True


In [5]:
import plotly.graph_objects as go

# Giả sử DataFrame của bạn tên là `data` và có chỉ mục datetime
fig = go.Figure()

# Thêm đường biểu diễn giá đóng cửa và giá dự đoán
fig.add_trace(go.Scatter(x=data.index, y=data['Close'], mode='lines', name='Close'))
fig.add_trace(go.Scatter(x=data.index, y=data['Predicted_Close'], mode='lines', name='Predicted Close'))

# Thêm điểm cho tín hiệu mua và bán
fig.add_trace(go.Scatter(x=data[data['Buy_Signal']].index, y=data[data['Buy_Signal']]['Close'], 
                         mode='markers', name='Buy Signal', marker=dict(color='green', size=10, symbol='triangle-up')))
fig.add_trace(go.Scatter(x=data[data['Sell_Signal']].index, y=data[data['Sell_Signal']]['Close'], 
                         mode='markers', name='Sell Signal', marker=dict(color='red', size=10, symbol='triangle-down')))

# Thêm đường cho Momentum và RSI với trục y phụ
fig.add_trace(go.Scatter(x=data.index, y=data['Momentum'], name='Momentum', yaxis='y2'))
fig.add_trace(go.Scatter(x=data.index, y=data['RSI'], name='RSI', yaxis='y3'))

# Tùy chỉnh layout cho các trục y phụ
fig.update_layout(
    title='Trading Signals with ARIMA, Momentum, and RSI',
    xaxis_title='Date',
    yaxis_title='Close Price',
    yaxis2=dict(
        title='Momentum',
        titlefont=dict(color='purple'),
        tickfont=dict(color='purple'),
        overlaying='y',
        side='right'
    ),
    yaxis3=dict(
        title='RSI',
        titlefont=dict(color='orange'),
        tickfont=dict(color='orange'),
        overlaying='y',
        side='right',
        position=0.95
    )
)

# Hiển thị biểu đồ
fig.show()


In [7]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Mã chuẩn bị dữ liệu ở đây
# Giả sử 'data' là DataFrame của bạn và nó có 'Datetime' làm chỉ mục

# Tạo biểu đồ con: 3 hàng cho Giá đóng cửa, Momentum, và RSI
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=('Trading Signals with ARIMA, Momentum, and RSI', 'Momentum', 'RSI'),
                    vertical_spacing=0.1)

# Thêm giá đóng và dự đoán giá đóng vào hàng đầu tiên
fig.add_trace(go.Scatter(x=data.index, y=data['Close'], mode='lines', name='Giá Đóng'), row=1, col=1)
fig.add_trace(go.Scatter(x=data.index, y=data['Predicted_Close'], mode='lines', name='Dự Đoán Giá Đóng'), row=1, col=1)

# Thêm tín hiệu mua và bán trên cùng một biểu đồ con
fig.add_trace(go.Scatter(x=data[data['Buy_Signal']].index, y=data[data['Buy_Signal']]['Close'], mode='markers',
                         marker=dict(color='green', size=10, symbol='triangle-up'), name='Tín Hiệu Mua'), row=1, col=1)
fig.add_trace(go.Scatter(x=data[data['Sell_Signal']].index, y=data[data['Sell_Signal']]['Close'], mode='markers',
                         marker=dict(color='red', size=10, symbol='triangle-down'), name='Tín Hiệu Bán'), row=1, col=1)

# Thêm Momentum vào hàng thứ hai
fig.add_trace(go.Scatter(x=data.index, y=data['Momentum'], mode='lines', name='Momentum'), row=2, col=1)

# Thêm RSI vào hàng thứ ba
fig.add_trace(go.Scatter(x=data.index, y=data['RSI'], mode='lines', name='RSI'), row=3, col=1)

# Cập nhật layout
fig.update_layout(height=800, title_text="Chiến Lược Giao Dịch với Tín Hiệu Mua/Bán, Momentum và RSI")

# Hiển thị biểu đồ
fig.show()


# Day tin hieu qua Redis

In [5]:
import pandas as pd
import plotly.graph_objects as go
import redis
import numpy as np
from datetime import datetime

# Nếu có tín hiệu thì đẩy qua Redis
# Datetime  Open    High    Low	Close   Volume  PredictClose Momentum RSI Buy_Signal  Sell_Signal
# Tạo kết nối Redis
r = redis.Redis(host='localhost', port=6379, db=11)
# Tạo hash key
hash_key = 'OG_FX_Arima, Momentum, RSI_K9'
last_record = data.iloc[-1]

# Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True):
    for field, value in last_record.to_dict().items():
        # Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
        if isinstance(value, pd.Timestamp):
            value = value.isoformat()
        elif isinstance(value, (int, np.uint64)):
            value = str(value)
        r.hset(hash_key, field, value)  
        r.hset(hash_key, 'Symbol', symbol)
        r.hset(hash_key, 'Insertdate', datetime.now().isoformat())
# Datetime  Open    High    Low	Close   Volume  PredictClose Momentum RSI Buy_Signal  Sell_Signal Symbol Insertdate
    print(last_record)   
else:
    print('Không có tín hiệu!')   

Open                   98056.05
High                   98147.45
Low                    97935.15
Close                  98072.08
Volume                      169
Momentum                  22.53
RSI                   77.001437
Predicted_Close    98053.997376
Buy_Signal                False
Sell_Signal                True
Name: 2024-11-29 17:00:00, dtype: object
